# Hardware-aware MCX synthesis

Write one all-ones control and let the compiler choose working storage and route it. Preserve15 controls, the width18 ceiling, depth objective and CX/U basis for both target connectivities. Counts describe static native programs, not hardware performance. No hosted viewer, source code loader or provider credentials are needed.


In [ ]:
import flagquantum as fq
from examples.synthesis.tutorials.technology_demonstrations.hardware_aware_mcx.overview.workflow import (
    run_example,
)
from flagquantum import synth
from flagquantum.compiler import ResourceEstimate
from flagquantum.synth import Output, QArray, QBit, X, allocate, control, qfunc


@qfunc
def main(cntrl: Output[QArray[QBit]], target: Output[QBit]) -> None:
    allocate(15, cntrl)
    allocate(target)
    control(cntrl, lambda: X(target))


qprog_linear = synth.synthesize(
    main,
    basis=("cx", "u"),
    topology="line",
    objective="depth",
    max_qubits=18,
    memory_limit_bytes=512 * 1024**2,
)
qprog_all_to_all = synth.synthesize(
    main,
    basis=("cx", "u"),
    objective="depth",
    max_qubits=18,
    memory_limit_bytes=512 * 1024**2,
)
print("Linear connectivity:", tuple((q, q + 1) for q in range(17)))
print("All-to-all resources:", qprog_all_to_all.report["resources"])
print("Linear resources:", qprog_linear.report["resources"])


## All-to-all depth

Read the actual normalized executable cost. The selected implementation uses one clean work bit. Native construction is deterministic; the source platform unspecified seed has no sampling counterpart here.


In [ ]:
all_to_all_resources = qprog_all_to_all.report["resources"]
assert isinstance(all_to_all_resources, ResourceEstimate)
print(all_to_all_resources.depth)


## Linear depth

The same logical control is routed onto adjacent wires. Extra CX gates implement exact routing SWAPs with retained layout evidence. These depths are native measurements; they do not reproduce the proprietary source engine optimization or establish a global optimum.


In [ ]:
linear_resources = qprog_linear.report["resources"]
assert isinstance(linear_resources, ResourceEstimate)
print(linear_resources.depth)


## Independent execution checks

Run both original all-zero programs and supplementary active, near-active, nonzero-target and coherent states. Independent Circuit preparation plus an explicit basis-index swap provides the truth oracle without sharing the synthesizer decomposition. Compare every complex amplitude, including absolute phase, and every hidden work amplitude. Read the target by name.


In [ ]:
checks = run_example()
print("Validated scenarios:", len(checks))
print("Largest absolute state error:", max(row.maximum_state_error for row in checks))
print(
    "Largest dirty workspace probability:",
    max(row.dirty_workspace_probability for row in checks),
)
print(
    "Named original target:",
    dict(qprog_linear.probabilities(fq.run(qprog_linear), registers=("target",))),
)
